In [31]:
import os, glob, random, csv
import numpy as np
import nibabel as nib
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import List, Dict, Tuple
from scipy.ndimage import binary_dilation, gaussian_filter
from torch.utils.data import ConcatDataset

from torch.optim import Adam, AdamW, lr_scheduler

from monai.transforms import (
    Compose, LoadImaged, ScaleIntensityd, AsDiscreted, CropForegroundd, RandSpatialCropd, EnsureChannelFirstd, EnsureTyped, Resized,
    Orientationd, Spacingd,
    ScaleIntensityRanged, NormalizeIntensityd,
    RandCropByPosNegLabeld, RandFlipd, SpatialPadd, 
    RandAffined, RandScaleIntensityd, RandGaussianNoised, RandGaussianSmoothd, MapTransform
)
from monai.data import Dataset, DataLoader, CacheDataset
from monai.networks.nets import UNet
from monai.losses import DiceCELoss, DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import Activations, AsDiscrete, ScaleIntensity
from monai.inferers import sliding_window_inference
from monai.transforms import SaveImaged

# Let Slurm control which GPU is visible on the node.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda:0


In [ ]:
# global parameters
OUT_DIR = "outputs/atlas1"
os.makedirs(OUT_DIR, exist_ok=True)

MAX_EPOCHS = 100
PATCH_SIZE = (128, 128, 128)
NUM_SAMPLES_PER_VOL = 8
LR = 1e-4

ROI_SIZE = (128, 128, 128)
SW_BATCH_SIZE = NUM_SAMPLES_PER_VOL

IN_CHANNELS = 1
OUT_CHANNELS = 1


In [33]:
# # =========================
# # 2) 快速检查 nii.gz 基本信息 + 可视化一张切片
# # =========================
# # assert os.path.exists(IMAGE_PATH) and os.path.exists(LABEL_PATH)
# # path = "atlas/T1/sub-005.nii.gz"
# path = "atlas/pathology_maps_segmentation/sub-001.nii.gz" 

# img_nii = nib.load(path)
# img = img_nii.get_fdata()
# print("Image shape:", img.shape)
# print(np.unique(img))
# # print("Affine:\n", img_nii.affine)

# z = img.shape[2] // 2  # 中间切片索引
# slice_img = img[:, :, z]

# plt.figure()
# plt.imshow(np.rot90(slice_img), cmap="gray")
# plt.title(f"Binary mid-slice z={z} (thr=0.5)")
# plt.axis("off")
# plt.show()

# # lab_nii = nib.load(train_data[10]['label'])
# # lab = lab_nii.get_fdata()
# # print("Label shape:", lab.shape)
# # z = 60
# # lab_img = lab[:, :, z]

# # l_min, l_max = float(lab_img.min()), float(lab_img.max())
# # lab_norm = (lab_img - l_min) / (l_max - l_min + 1e-8)
# # lab_bin = (lab_norm > 0.5).astype(np.uint8)

# # plt.figure()
# # plt.imshow(np.rot90(lab_bin), cmap="gray")
# # plt.title(f"Label mid-slice z={z}")
# # plt.axis("off")
# # plt.show()


In [34]:
class GaussianThresholdBackgroundd(MapTransform):
    def __init__(self, keys, label_key="label", sigma=0.5, thr_ratio=0.02, use_abs=False, margin=1):
        super().__init__(keys)
        self.label_key = label_key
        self.sigma = sigma
        self.thr_ratio = thr_ratio
        self.use_abs = use_abs
        self.margin = margin

    def __call__(self, data):
        d = dict(data)

        img = d[self.keys[0]]
        label = d[self.label_key]

        is_tensor = torch.is_tensor(img)

        if is_tensor:
            img_np = img.detach().cpu().numpy().copy()
            lab_np = label.detach().cpu().numpy()
        else:
            img_np = np.asarray(img).copy()
            lab_np = np.asarray(label)

        label_mask = lab_np > 0.5   # shape: [C, H, W, D]

        image_mask = np.zeros_like(img_np, dtype=bool)

        for c in range(img_np.shape[0]):
            x = img_np[c]

            x_proc = np.abs(x) if self.use_abs else x
            smoothed = gaussian_filter(x_proc, sigma=self.sigma)

            thr = float(smoothed.max()) * self.thr_ratio
            image_mask[c] = smoothed >= thr

        final_mask = image_mask | label_mask # combine image and label masks

        # reserve mask margin
        if self.margin > 0:
            structure = np.ones((3, 3, 3), dtype=bool)

            for c in range(final_mask.shape[0]):
                final_mask[c] = binary_dilation(
                    final_mask[c],
                    structure=structure,
                    iterations=self.margin
                )

        img_np = np.where(final_mask, img_np, 0.0)

        if is_tensor:
            d[self.keys[0]] = torch.as_tensor(img_np, dtype=img.dtype, device=img.device)
        else:
            d[self.keys[0]] = img_np

        return d

class AlignAxesd(MapTransform):
    def __init__(self, keys=["image", "label"], transpose_order=(0, 2, 1), flip_axes=[2]):
        super().__init__(keys)
        self.transpose_order = transpose_order
        self.flip_axes = flip_axes if flip_axes is not None else []

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            img = d[key]

            if img.ndim == 4:
                c = img.shape[0]
                spatial = img.shape[1:]

                if self.transpose_order is not None:
                    img = np.transpose(img, (0,) + tuple(i + 1 for i in self.transpose_order))

                for ax in self.flip_axes:
                    img = np.flip(img, axis=ax + 1)

            elif img.ndim == 3:
                if self.transpose_order is not None:
                    img = np.transpose(img, self.transpose_order)

                for ax in self.flip_axes:
                    img = np.flip(img, axis=ax)

            else:
                raise ValueError(f"{key} shape {img.shape} is not supported")

            d[key] = img.copy()

        return d

In [35]:
common_keys = ["image", "label"]

common = [
    LoadImaged(keys=common_keys, image_only=False),
    EnsureChannelFirstd(keys=common_keys),
    Orientationd(keys=common_keys, axcodes="RAS"),
    Spacingd(keys=common_keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
]

crop_resize_norm = [
    CropForegroundd(keys=common_keys, source_key="image", margin=10),
    # Resized(keys=common_keys, spatial_size=(128, 128, 128)),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
]

rand_crop = [
    RandCropByPosNegLabeld(
        keys=common_keys,
        label_key="label",
        spatial_size=PATCH_SIZE,
        pos=1, neg=1,
        num_samples=NUM_SAMPLES_PER_VOL,
        image_key="image",
        allow_smaller=True,
    )
]

augmentation = [
    RandFlipd(keys=common_keys, prob=0.5, spatial_axis=0),
    RandFlipd(keys=common_keys, prob=0.5, spatial_axis=1),
    RandFlipd(keys=common_keys, prob=0.5, spatial_axis=2),
    RandAffined(
        keys=common_keys, prob=0.3,
        rotate_range=(0.3, 0.3, 0.3), scale_range=(0.1, 0.1, 0.1),
        mode=("bilinear", "nearest"), padding_mode="zeros",
    ),
    RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.3),
    
    # RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.05),
    # RandGaussianSmoothd(
    #     keys=["image"], prob=0.2,
    #     sigma_x=(0.5, 1.5), sigma_y=(0.5, 1.5), sigma_z=(0.5, 1.5),
    # ),
]

gen_tf = Compose(common + [
    AlignAxesd(keys=common_keys, transpose_order=(0, 2, 1), flip_axes=[2]), # align axes
    AsDiscreted(keys=["label"], threshold=0.5, to_onehot=None, dtype=np.float32),
    GaussianThresholdBackgroundd(keys=["image"], sigma=0.5, thr_ratio=0.02, margin=1),
] + crop_resize_norm + rand_crop + augmentation + [
    EnsureTyped(keys=common_keys),
])

train_tf = Compose(common + crop_resize_norm + rand_crop + augmentation + [
    EnsureTyped(keys=common_keys),
])

val_tf = Compose(common + crop_resize_norm + [
    EnsureTyped(keys=common_keys),
])

post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])

/home/kxu56/.conda/envs/monai/lib/python3.10/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [36]:
class DatasetBuilder:
    """
    Build train / val data list for:
    - ISLES-2022
    - optional uncond_gen synthetic data
    """

    def __init__(
        self,
        real_root: str,
        gen_root: str = "uncond_gen",
        seed: int = 42,
        train_ratio: float = 0.8,
        filter: bool = True,
        gen_ratio: float = 0.0,
    ):
        self.real_root = real_root
        self.gen_root = gen_root
        self.seed = seed
        self.train_ratio = train_ratio
        self.filter = filter # whether filter out cases with empty foreground
        self.gen_ratio = gen_ratio

    # ----------------------------------------------------
    # Build real dataset list
    # ----------------------------------------------------
    def build_real_list(self) -> List[Dict]:
        data_list = []

        img_paths = glob.glob(
            os.path.join(
                self.real_root,
                "sub-strokecase*/ses-0001/dwi/*_dwi.nii.gz",
            )
        )

        for img in img_paths:
            case_id = os.path.basename(img).split("_")[0]

            lab = os.path.join(
                self.real_root,
                "derivatives",
                case_id,
                "ses-0001",
                f"{case_id}_ses-0001_msk.nii.gz",
            )

            if os.path.exists(lab):
                data_list.append({"image": img, "label": lab})

        print(f"[ISLES] Found {len(data_list)} cases")
        return data_list

    # ----------------------------------------------------
    # Filter foreground
    # ----------------------------------------------------
    def filter_foreground(self, data_list: List[Dict]) -> List[Dict]:
        if not self.filter:
            return data_list

        filtered = []
        skipped = 0

        for item in tqdm(data_list, desc="Filtering foreground"):
            lab = nib.load(item["label"]).get_fdata()

            if np.any(lab > 0):
                filtered.append(item)
            else:
                skipped += 1

        print(f"[Filter] keep {len(filtered)} cases, skip {skipped}")
        return filtered

    # ----------------------------------------------------
    # Split train / val
    # ----------------------------------------------------
    def split_train_val(
        self, data_list: List[Dict]
    ) -> Tuple[List[Dict], List[Dict]]:

        random.seed(self.seed)
        random.shuffle(data_list)

        n = len(data_list)
        split_idx = int(self.train_ratio * n)

        train_data = data_list[:split_idx]
        val_data = data_list[split_idx:]

        print(f"[Split] train={len(train_data)}, val={len(val_data)}")
        return train_data, val_data

    # ----------------------------------------------------
    # Add uncond synthetic data
    # ----------------------------------------------------
    def add_gen(
        self, train_data: List[Dict],
    ) -> List[Dict]:
        len_train_data = len(train_data)

        img_paths = glob.glob(
            os.path.join(self.gen_root, "y0_*.nii.gz")
        )

        count = 0
        for img in img_paths:
            if count >= int(self.gen_ratio * len_train_data):
                break
            case_id = os.path.basename(img).split("_")[1]
            lab = os.path.join(self.gen_root, f"x0_{case_id}")

            if os.path.exists(lab):
                train_data.append({"image": img, "label": lab})
                count += 1

        print(f"[Uncond] Added {count} synthetic samples")
        return train_data

    # ----------------------------------------------------
    # Build final lists
    # ----------------------------------------------------
    def build(self) -> Tuple[List[Dict], List[Dict]]:
        real_list = self.build_real_list()
        real_list = self.filter_foreground(real_list)
        train_data, val_data = self.split_train_val(real_list)
        train_data = self.add_gen(train_data)
        return train_data, val_data


In [37]:
class AtlasBuilder(DatasetBuilder):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # self.real_root = "atlas"

    def build_real_list(self) -> List[Dict]:
        data_list = []

        img_paths = glob.glob(
            os.path.join(
                self.real_root,
                "T1/sub-*.nii.gz",
            )
        )
        
        for img in tqdm(img_paths, desc="Building real list"):
            case_id = os.path.basename(img)

            lab = os.path.join(
                self.real_root,
                "pathology_maps_segmentation",
                case_id,
            )

            if os.path.exists(lab):
                data_list.append({"image": img, "label": lab})

        print(f"Found {len(data_list)} real cases")
        return data_list
    
    def build_gen_list(self, train_data) -> List[Dict]:
        data_list = []
        count = self.gen_ratio * len(train_data)

        img_paths = glob.glob(
            os.path.join(
                self.gen_root,
                "y0_*.nii.gz",
            )
        )

        for i, img in enumerate(tqdm(img_paths, desc="Building synthetic list")):
            if i >= count:
                break
            case_id = os.path.basename(img).split(".")[0].split("_")[1]

            lab = os.path.join(
                self.gen_root,
                f"x0_{case_id}.nii.gz",
            )

            if os.path.exists(lab):
                data_list.append({"image": img, "label": lab})
                
        print(f"Found {len(data_list)} synthetic cases")
        return data_list
    
    def build(self) -> Tuple[List[Dict], List[Dict], List[Dict]]:
        real_data = self.build_real_list()
        # real_data = self.filter_foreground(real_data)
        train_data, val_data = self.split_train_val(real_data)
        # train_data = self.add_gen(train_data)
        gen_data = self.build_gen_list(train_data)
        return train_data, val_data, gen_data


In [ ]:
builder = AtlasBuilder(
    real_root="atlas",
    gen_root="uncond_gen",
    seed=42,
    train_ratio=0.8,
    filter=True,
    gen_ratio=0,
)
train_data, val_data, gen_data = builder.build()
# gen_data = builder.build_gen_list()

Building real list: 100%|██████████| 655/655 [00:00<00:00, 10592.66it/s]

Found 655 real cases
[Split] train=524, val=131



Building synthetic list:  11%|█         | 53/500 [00:00<00:00, 9025.50it/s]

Found 53 synthetic cases


In [39]:
gen_ds = CacheDataset(
    data=gen_data,
    transform=gen_tf,
    cache_rate=1.0,
    num_workers=16
)
real_ds = CacheDataset(
    data=train_data,
    transform=train_tf,
    cache_rate=1.0,
    num_workers=16
)
val_ds = CacheDataset(
    data=val_data,
    transform=val_tf,
    cache_rate=1.0,
    num_workers=16
)

train_ds = ConcatDataset([real_ds, gen_ds])
# train_ds = real_ds

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=16)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=16)

Loading dataset: 100%|██████████| 131/131 [00:47<00:00,  2.76it/s]


In [40]:
# import numpy as np
# import torch
# from monai.transforms import Compose

# def _to_np(x):
#     if torch.is_tensor(x):
#         return x.detach().cpu().numpy()
#     return np.asarray(x)

# def stats_tensor(arr, name):
#     arr = _to_np(arr)
#     print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, "
#           f"min={float(arr.min()):.4g}, max={float(arr.max()):.4g}, "
#           f"mean={float(arr.mean()):.4g}, std={float(arr.std()):.4g}")

# def stats_label(lab):
#     lab = _to_np(lab)
#     uniq = np.unique(lab)
#     print(f"label unique (up to 20): {uniq[:20]}  (n_unique={len(uniq)})")
#     print(f"label sum: {float(lab.sum()):.4g}, label mean: {float(lab.mean()):.4g}")
#     is_binary = set(uniq.tolist()).issubset({0, 1})
#     print("label is binary {0,1} ?", is_binary)

# def check_one_case(ds, idx=0):
#     raw = ds[idx]
#     # print("="*80)
#     # print(f"CASE idx={idx}")
#     # print("raw keys:", list(raw.keys()))

#     # AFTER common
#     # out_common = Compose(common)(raw)
#     # print("\n- AFTER common -")
#     # stats_tensor(out_common["image"], "image(common)")
#     # stats_tensor(out_common["label"], "label(common)")
#     # stats_label(out_common["label"])

#     # AFTER val_tf
#     # out_val = val_tf(raw)
#     # print("\n- AFTER val_tf -")
#     # stats_tensor(out_val["image"], "image(val_tf)")
#     # stats_tensor(out_val["label"], "label(val_tf)")
#     # stats_label(out_val["label"])

#     # AFTER train_tf
#     out_tr = train_tf(raw)
#     # print("\n- AFTER train_tf -")
#     # print("train_tf output type:", type(out_tr))

#     # 情况1：num_samples>1 -> list[dict]
#     if isinstance(out_tr, list):
#         # print("num_patches:", len(out_tr))
#         # 每个 patch 的 label.sum()，看正样本比例
#         sums = []
#         for i, p in enumerate(out_tr):
#             lab = _to_np(p["label"])
#             sums.append(float(lab.sum()))
#             # if i < 2:  # 只展示前2个 patch 的统计
#             #     stats_tensor(p["image"], f"patch[{i}].image")
#             #     stats_tensor(p["label"], f"patch[{i}].label")
#         pos_ratio = float(np.mean(np.array(sums) > 0))
#         # print("patch label sums (first 10):", sums[:10])
#         # print("pos patch ratio (label.sum>0):", pos_ratio)
#         return pos_ratio

#     # 情况2：num_samples==1 -> dict
#     elif isinstance(out_tr, dict):
#         stats_tensor(out_tr["image"], "image(train_tf)")
#         stats_tensor(out_tr["label"], "label(train_tf)")
#         stats_label(out_tr["label"])

#     else:
#         print("Unexpected output type:", type(out_tr))


# # 运行示例（把 train_ds/val_ds 换成你的 dataset）
# pos_ratio = []
# for idx in range(len(train_ds)):
#     pos_ratio.append(check_one_case(train_data, idx=idx))
#     if idx % 50 == 0:
#         print(f"Checking case idx={idx}...")
#         print(np.mean(pos_ratio))
# print(np.mean(pos_ratio))
# # check_one_case(train_data, idx=6)


In [41]:
# batch = next(iter(train_loader))

# print(batch.keys())
# print(batch["image"].shape)
# print(batch["label"].shape)

# img = batch["image"][0]   # [1, H, W, D]
# lab = batch["label"][0]

# img = img.squeeze(0)  # [H, W, D]
# lab = lab.squeeze(0)

# z = img.shape[-1] // 2

# plt.figure(figsize=(6,6))
# plt.imshow(img[:,:,z].cpu(), cmap="gray")
# plt.imshow(lab[:,:,z].cpu(), alpha=0.4, cmap="Reds")
# plt.title("Overlay")
# plt.show()

# plt.show()


In [42]:
# define model
model = UNet(
    spatial_dims=3,
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

print("Params:", sum(p.numel() for p in model.parameters())/1e6, "M")

Params: 4.805534 M


In [43]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="Using a non-tuple sequence for multidimensional indexing is deprecated",
)

warnings.filterwarnings(
    "ignore",
    message=".*Num foregrounds.*",
)


In [44]:
log_path = os.path.join(OUT_DIR, "training_log.csv")

if not os.path.exists(log_path):
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_loss",
            "val_loss",
            "train_dice",
            "val_dice",
            "lr"
        ])


In [45]:
# train and validate

load_best = False

best_path = glob.glob(os.path.join(OUT_DIR, "unet3d_best_*.pt"))
if len(best_path) > 0:
    best_path = best_path[0]
else:
    best_path = 'None'

last_path = glob.glob(os.path.join(OUT_DIR, "unet3d_last_*.pt"))
if len(last_path) > 0:
    last_path = last_path[0]
else:
    last_path = 'None'

log_path = os.path.join(OUT_DIR, "training_log.csv")

# loss_fn = DiceLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)
# loss_fn = DiceLoss(to_onehot_y=False, sigmoid=True)
loss_fn = DiceCELoss(to_onehot_y=False, sigmoid=True)
optimizer = Adam(model.parameters(), lr=LR)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

if os.path.exists(last_path) and os.path.exists(best_path): 
    best_epoch = int(best_path.split('_')[-1].split('.')[0])
    last_epoch = int(last_path.split('_')[-1].split('.')[0])
    start_epoch = last_epoch

    assert os.path.exists(log_path), "Log file not found!"

    with open(log_path, "r") as f:
        # read the best epoch
        reader = csv.DictReader(f)
        lines = list(reader)
        if len(lines) > 1:
            best_line = lines[best_epoch-1]
            best_dice = float(best_line["val_dice"])

    if load_best:
        ckpt = torch.load(best_path, map_location=device)
        start_epoch = int(ckpt["epoch"])
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["lr_scheduler"])
        print("Loaded best model:", best_path, f"from epoch {best_epoch}")
    else:
        ckpt = torch.load(last_path, map_location=device)
        start_epoch = int(ckpt["epoch"])
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["lr_scheduler"])
        print("Loaded last model:", last_path, f"from epoch {last_epoch}")
    
else:
    start_epoch = 0
    best_dice = -1.0
    print("No model found, training from scratch.")

dice_metric = DiceMetric(include_background=False, reduction="mean")

# train_dice_list = []
# train_loss_list = []
# val_dice_list = []
# val_loss_list = []

for epoch in range(start_epoch, MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]")
    current_lr = optimizer.param_groups[0]["lr"] # get lr 
    for i, batch in enumerate(pbar):
        y = batch["image"].to(device)
        x = batch["label"].to(device)
        # print("y shape:", y.shape)  # (B, C, D, H, W)

        # assert (np.unique(x) == [0,1]).all(), f'{np.unique(x)}'
        assert x.shape == y.shape

        logits = model(y)
        loss = loss_fn(logits, x)
        
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        train_avg_loss = epoch_loss / (i + 1)

        x_pred = post_trans(logits)
        dice_metric(x_pred, x)
        # dice = dice_metric.aggregate().item()

        pbar.set_postfix({
            "avg_loss": f"{train_avg_loss: .4f}",
            "lr": f"{current_lr:.6e}"
        })

    train_avg_dice = dice_metric.aggregate().item()
    dice_metric.reset()
    # train_dice_list.append(train_avg_dice) # record train dice for this epoch
    # train_loss_list.append(train_avg_loss) # average batch loss for this epoch

    model.eval()
    epoch_loss = 0.0
    pbar = tqdm(val_loader, desc="[Val]")
    with torch.no_grad():
        for i, batch in enumerate(pbar):
            y = batch["image"].to(device)
            x = batch["label"].to(device)

            logits = sliding_window_inference(y, ROI_SIZE, SW_BATCH_SIZE, model)
            loss = loss_fn(logits, x)
            epoch_loss += loss.item()
            val_avg_loss = epoch_loss / (i + 1)

            x_pred = post_trans(logits)
            # print(x_pred.shape, x.shape)
            # assert x_pred.shape == x.shape
            dice_metric(x_pred, x)
            # dice = dice_metric.aggregate().item() # accumulated dice       

            pbar.set_postfix({
                "avg_loss": f"{val_avg_loss: .4f}"
            })

    val_avg_dice = dice_metric.aggregate().item()   
    dice_metric.reset()
    # val_dice_list.append(val_avg_dice)
    # val_loss_list.append(val_avg_loss)
    print(f"Epoch {epoch+1} train dice: {train_avg_dice:.4f}, val dice: {val_avg_dice:.4f}")

    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch + 1,
            f"{train_avg_loss:.4f}",
            f"{val_avg_loss:.4f}",
            f"{train_avg_dice:.4f}",
            f"{val_avg_dice:.4f}",
            f"{current_lr:.6e}",
        ])
    scheduler.step() # update lr

    if val_avg_dice > best_dice:
        best_dice = val_avg_dice
        new_best_path = os.path.join(OUT_DIR, f"unet3d_best_{epoch + 1}.pt")
        torch.save({"epoch": epoch + 1,
            "epoch": epoch + 1,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "lr_scheduler": scheduler.state_dict(),
        }, new_best_path) # save new best model
        if os.path.exists(best_path):
            os.remove(best_path) # remove old best model
        best_path = new_best_path
        print("saved best ->", best_path)

    if (epoch + 1) % 5 == 0: # save last model every 5 epochs
        new_last_path = os.path.join(OUT_DIR, f"unet3d_last_{epoch + 1}.pt")
        torch.save({
            "epoch": epoch + 1,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "lr_scheduler": scheduler.state_dict(),
        }, new_last_path) # save new last model
        if os.path.exists(last_path):
            os.remove(last_path) # remove old last model
        last_path = new_last_path
        print("saved last ->", last_path)

new_last_path = os.path.join(OUT_DIR, f"unet3d_last_{epoch + 1}.pt")
torch.save({
    "epoch": epoch + 1,
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "lr_scheduler": scheduler.state_dict(),
}, new_last_path) # save new last model
# if os.path.exists(last_path):
#     os.remove(last_path) # remove old last model
#     last_path = new_last_path
#     print("saved last ->", last_path)

print("Training done. best dice:", best_dice)


No model found, training from scratch.


[Val]: 100%|██████████| 131/131 [00:33<00:00,  3.86it/s, avg_loss=1.6460]


Epoch 1 train dice: 0.0415, val dice: 0.0314
saved best -> outputs/atlas12+uncond1/unet3d_best_1.pt


[Val]: 100%|██████████| 131/131 [00:34<00:00,  3.74it/s, avg_loss=1.6285]


Epoch 2 train dice: 0.0411, val dice: 0.0099


...

Epoch 98 train dice: 0.6794, val dice: 0.5502


[Val]: 100%|██████████| 131/131 [00:26<00:00,  4.89it/s, avg_loss=0.4831]


Epoch 99 train dice: 0.6780, val dice: 0.5480


[Val]: 100%|██████████| 131/131 [00:26<00:00,  4.90it/s, avg_loss=0.4799]


Epoch 100 train dice: 0.6777, val dice: 0.5513
saved last -> outputs/atlas12+uncond1/unet3d_last_100.pt
Training done. best dice: 0.5609567165374756
